# 07 Supervisor Agent

Run the notebook-facing supervisor service. The supervisor calls the RAG-based fundamental worker, technical chart worker, and Tavily news worker, then aggregates their 1-100 ratings into a final future-perspective rating.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from market_analyst.config.settings import load_settings
from market_analyst.services.supervisor import run_supervisor_agent
from market_analyst.telemetry import configure_notebook_logging
from market_analyst.types.supervisor import SupervisorAnalysisRequest

logger = configure_notebook_logging(run_name="07_supervisor_agent")
settings = load_settings()

settings.require_chat_model()
settings.require_database()
settings.require_embeddings()
settings.require_tavily()

print("Project root:", PROJECT_ROOT)
print("Chat deployment:", settings.azure_openai_chat_deployment)
print("Vector collection:", settings.vector_collection_name)
print("Tavily configured:", bool(settings.tavily_api_key))

c:\Users\rushi\OneDrive - ImmersiLearn Education Services LLP\Projects\LLM Projects\market-analyst-feb26\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026-05-16 10:52:56,708 INFO 07_supervisor_agent notebook_run_started


Project root: c:\Users\rushi\OneDrive - ImmersiLearn Education Services LLP\Projects\LLM Projects\market-analyst-feb26
Chat deployment: gpt-5.4-mini
Vector collection: fundamental_report_chunks
Tavily configured: True


## Run Configuration

Set the sample company inputs. The RAG store should already contain annual-report chunks from `03_rag_pipeline.ipynb` or the shared backend ingestion path. The technical worker will fetch price history for the ticker, and the news worker will run current company plus sector searches.

In [6]:
COMPANY_NAME = "Bandhan Bank"
TICKER = "BANDHANBNK.NS"
SECTOR = None

request = SupervisorAnalysisRequest(
    company_name=COMPANY_NAME,
    ticker=TICKER,
    sector=SECTOR,
)

print(request)

SupervisorAnalysisRequest(company_name='Bandhan Bank', ticker='BANDHANBNK.NS', sector=None, fundamental_question=None, technical_question=None, news_question=None)


## Run The Supervisor

The call below uses the shared supervisor service. Worker prompts and score parsing stay in reusable modules; the notebook only configures inputs and displays the result.

In [7]:
result = run_supervisor_agent(settings, request)

print(result.summary)

2026-05-16 11:04:28,054 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/gpt-5.4-mini/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-16 11:04:37,920 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-16 11:04:43,086 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/gpt-5.4-mini/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-16 11:04:53,159 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-16 11:04:53,393 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=

Bandhan Bank (BANDHANBNK.NS) final future-perspective rating is 61/100. Component ratings: fundamental=61, technical=68, news=52.


## Inspect Worker Ratings

This cell prints the worker ratings and rationales that produced the final supervisor rating.

In [9]:
for component in result.components:
    print(f"{component.name}: rating={component.rating}, weight={component.weight:.2f}")
    print(component.rationale[:500])
    print()

fundamental: rating=61, weight=0.45
{ "company_name": "Bandhan Bank", "ticker": "BANDHANBNK", "fundamental_rating": 61, "growth": [ "FY25 total net revenue rose 15.7% to ₹14,457.18 crore, and net interest income grew 11.4% to ₹11,490.58 crore.", "PAT increased 23% year-on-year despite a challenging macro backdrop a...

technical: rating=68, weight=0.30
{ "ticker": "BANDHANBNK.NS", "technical_rating": 68, "trend": "Medium-term uptrend remains intact, but the latest pullback has softened the short-term trend. Price is still above the 50-day moving average and well above the 20-day MA's recent base, though the close has slipped fr...

news: rating=52, weight=0.25
{"company_name":"Bandhan Bank","ticker":"BANDHANBNK.NS","sector":"Banking (private lender / retail & microfinance-focused)","rating":52,"sentiment_score":51,"positive_developments":["Recent Indian bank results from peers such as ICICI Bank and RBL Bank showed stronger loan growth...



## Validation

In [ ]:
assert 1 <= result.final_rating <= 100
assert len(result.components) == 3
assert {component.name for component in result.components} == {"fundamental", "technical", "news"}

print("Supervisor notebook validation passed.")

## Supervisor Chat Follow-Up Test

This section validates the chat-facing supervisor layer. It reuses the static `result` snapshot above, keeps short-term chat history in the notebook, and lets the supervisor chat agent call the fundamental, technical, or news worker tools when the user asks a follow-up question.

In [10]:
from market_analyst.services.supervisor_chat import run_supervisor_chat_turn
from market_analyst.types.supervisor_chat import SupervisorChatContext, SupervisorChatRequest

chat_context = SupervisorChatContext(
    company_name=request.company_name,
    ticker=request.ticker,
    sector=request.sector,
    supervisor_result=result,
)

chat_history = []
print(chat_context)

SupervisorChatContext(company_name='Bandhan Bank', ticker='BANDHANBNK.NS', sector=None, supervisor_result=SupervisorAnalysisResult(company_name='Bandhan Bank', ticker='BANDHANBNK.NS', final_rating=61, summary='Bandhan Bank (BANDHANBNK.NS) final future-perspective rating is 61/100. Component ratings: fundamental=61, technical=68, news=52.', components=[SupervisorRatingComponent(name='fundamental', rating=61, weight=0.45, rationale='{ "company_name": "Bandhan Bank", "ticker": "BANDHANBNK", "fundamental_rating": 61, "growth": [ "FY25 total net revenue rose 15.7% to ₹14,457.18 crore, and net interest income grew 11.4% to ₹11,490.58 crore.", "PAT increased 23% year-on-year despite a challenging macro backdrop a...'), SupervisorRatingComponent(name='technical', rating=68, weight=0.3, rationale='{ "ticker": "BANDHANBNK.NS", "technical_rating": 68, "trend": "Medium-term uptrend remains intact, but the latest pullback has softened the short-term trend. Price is still above the 50-day moving ave

## Ask A Supervisor-Level Follow-Up

This first turn should usually answer from the attached supervisor snapshot. The model can still call worker tools if it needs more evidence.

In [11]:
chat_request = SupervisorChatRequest(
    context=chat_context,
    message="What is the main reason behind the final rating, and which area should I inspect first?",
    history=chat_history,
)

chat_response = run_supervisor_chat_turn(settings, chat_request)
chat_history = chat_response.history

print(chat_response.answer)
print("\nTools used:", chat_response.tool_names)
print("History length:", len(chat_history))

2026-05-16 12:24:51,288 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/gpt-5.4-mini/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-16 12:24:53,296 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/gpt-5.4-mini/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-16 12:25:03,180 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-16 12:25:07,803 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/gpt-5.4-mini/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-16 12:25:17,720 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2025

The main reason behind the **61/100 rating** is a **mixed profile**:

- **Positives:** solid growth and profitability, plus strong capitalization  
  - Revenue up **15.7%**
  - PAT up **23%**
  - Capital adequacy at **18.7%**
- **Negatives:** credit-quality and efficiency pressure, especially in microfinance  
  - Higher **credit costs**
  - Elevated **GNPA/NNPA**
  - Softer **NIM**
  - Worse **cost-to-income ratio**

### What to inspect first
Start with **asset quality and microfinance stress**.  
That is the main area holding the rating back, and it likely explains most of the gap between Bandhan Bank’s growth/capital strength and its only moderate overall score.

If you want, I can also break down the rating into **fundamental vs technical vs news** and show which one is dragging or supporting the total most.

Tools used: ['ask_fundamental_agent']
History length: 2


## Ask A Tool-Routed Follow-Up

This turn asks specifically for a technical view, so the supervisor chat agent should be able to call the technical worker tool while preserving the Yahoo Finance ticker exactly as provided.

In [ ]:
tool_request = SupervisorChatRequest(
    context=chat_context,
    message="Ask the technical agent whether the current momentum supports the supervisor rating.",
    history=chat_history,
)

tool_response = run_supervisor_chat_turn(settings, tool_request)
chat_history = tool_response.history

print(tool_response.answer)
print("\nTools used:", tool_response.tool_names)
print("History length:", len(chat_history))

## Chat Validation

In [ ]:
assert chat_response.answer.strip(), "Supervisor chat response should not be empty."
assert tool_response.answer.strip(), "Tool-routed supervisor chat response should not be empty."
assert len(chat_history) >= 4, "Chat history should include both user/assistant turns."
assert chat_history[-1].role == "assistant"

print("Supervisor chat notebook validation passed.")